# 03 — Environmental Factors (Member 2)
**Project:** Predicting Road Accident Severity Using Data Science — Milestone 1 · Report section **IV-E**
**Owner:** Member 2 · **Input:** `collisions_2024_2025_prepared.csv.gz` (from `01_data_preparation.ipynb`)

Question: is the KSI rate (killed or seriously injured vs slight) related to **light, weather and road surface**, and does the pattern hold in both years?

Steps: (1) load · (2) KSI rate tables · (3) chi-square + Cramér's V · (4) pairwise z-tests · (5) darkness vs urban/rural · (6) recorded vs severity-adjusted year change · (7) save **Fig. 3**.
Unknown values are excluded from each test (listwise, per factor).

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/Road_Accident_Project"   # change if your folder has another name
except ImportError:
    DATA_DIR = "."

df = pd.read_csv(os.path.join(DATA_DIR, "collisions_2024_2025_prepared.csv.gz"), low_memory=False)
df = df.replace({"Unknown": np.nan})          # safety: treat 'Unknown' as missing
print(f"{len(df):,} collisions | overall KSI rate = {df['KSI'].mean()*100:.2f}%")

## Step 2 — KSI rate by light, weather and road surface

In [ ]:
FACTORS = {"light_group": "Light (2 groups)", "darkness_lighting": "Light (detailed)",
           "weather_group": "Weather", "surface_group": "Road surface"}

def rate_table(col):
    d = df.dropna(subset=[col])
    t = d.groupby(col).agg(n=("KSI", "size"), KSI_pct=("KSI", lambda s: s.mean()*100),
                           KSI_adj_pct=("KSI_adjusted", lambda s: s.mean()*100))
    y = d.groupby([col, "collision_year"])["KSI"].mean().unstack()*100
    t["KSI_2024"], t["KSI_2025"] = y[2024], y[2025]
    return t.round(2)

for col, name in FACTORS.items():
    print(f"\n=== {name} ===")
    print(rate_table(col).to_string())

## Step 3 — Chi-square test of independence and Cramér's V
Yates' correction is used for 2 × 2 tables (as stated in the report's Methods). We also check that every expected count is at least 5.

In [ ]:
def chi_square(col, data=None):
    d = (df if data is None else data).dropna(subset=[col])
    ct = pd.crosstab(d[col], d["KSI"])
    chi2, p, dof, expected = stats.chi2_contingency(ct, correction=(ct.shape == (2, 2)))
    n = ct.values.sum()
    v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    return dict(factor=col, n=n, chi2=round(chi2, 1), dof=dof, p=p, cramers_V=round(v, 3),
                min_expected=round(expected.min(), 1))

res = pd.DataFrame([chi_square(c) for c in ["light_group", "weather_group", "surface_group", "darkness_lighting"]])
res["p"] = res["p"].map(lambda x: "< 0.001" if x < 0.001 else f"{x:.3f}")
res

## Step 4 — Pairwise two-proportion z-tests (two-sided)

In [ ]:
def ztest(col, a, b):
    A = df.loc[df[col] == a, "KSI"]; B = df.loc[df[col] == b, "KSI"]
    p = (A.sum() + B.sum()) / (len(A) + len(B))
    z = (A.mean() - B.mean()) / np.sqrt(p * (1 - p) * (1/len(A) + 1/len(B)))
    pval = 2 * (1 - stats.norm.cdf(abs(z)))
    return dict(comparison=f"{a} vs {b}", rate_a=round(A.mean()*100, 2), rate_b=round(B.mean()*100, 2),
                n_a=len(A), n_b=len(B), z=round(z, 2), p=round(pval, 3))

pairs = [("light_group", "Darkness", "Daylight"),
         ("darkness_lighting", "Dark - unlit/none", "Dark - lit"),
         ("weather_group", "Rain", "Fine"), ("weather_group", "Fog or mist", "Fine"),
         ("surface_group", "Wet or damp", "Dry"), ("surface_group", "Snow or ice", "Dry")]
pd.DataFrame([ztest(*x) for x in pairs])

## Step 5 — Is the darkness effect only a rural effect?
Unlit roads are mostly rural, so darkness could just be a proxy for location. We compare darkness with daylight **inside** urban and rural areas.

In [ ]:
d = df.dropna(subset=["light_group", "area"])
print((d.groupby(["area", "light_group"])["KSI"].agg(n="size", KSI_pct=lambda s: round(s.mean()*100, 1))).to_string())

d2 = df.dropna(subset=["darkness_lighting", "area"])
share = (d2.assign(unlit=d2["darkness_lighting"].eq("Dark - unlit/none"))
           .groupby("area")["unlit"].mean()*100).round(1)
print("\nShare of collisions on unlit dark roads (%):", share.to_dict())

## Step 6 — Year comparison: recorded vs severity-adjusted KSI
`KSI_adjusted` removes (as far as DfT can) the effect of the change to injury-based reporting.

In [ ]:
d = df.dropna(subset=["light_group"])
g = d.groupby(["light_group", "collision_year"]).agg(recorded=("KSI", "mean"), adjusted=("KSI_adjusted", "mean")) * 100
out = g.unstack()
out[("rise_pts", "recorded")] = out[("recorded", 2025)] - out[("recorded", 2024)]
out[("rise_pts", "adjusted")] = out[("adjusted", 2025)] - out[("adjusted", 2024)]
out.round(2)

## Step 7 — Fig. 3: KSI rate by light, weather and road surface (2024 vs 2025)

In [ ]:
plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "DejaVu Serif"], "font.size": 8})
overall = df["KSI"].mean() * 100

PANELS = [("(a) Light conditions", "darkness_lighting",
           [("Daylight", "Daylight"), ("Dark - lit", "Dark\n(lit)"),
            ("Dark - unlit/none", "Dark\n(unlit)"), ("Dark - unknown", "Dark\n(unknown)")]),
          ("(b) Weather", "weather_group",
           [("Fine", "Fine"), ("Rain", "Rain"), ("Snow", "Snow"), ("Fog or mist", "Fog/\nmist"), ("Other", "Other")]),
          ("(c) Road surface", "surface_group",
           [("Dry", "Dry"), ("Wet or damp", "Wet/\ndamp"), ("Snow or ice", "Snow/\nice"), ("Other hazard", "Other\nhazard")])]

fig, axs = plt.subplots(1, 3, figsize=(7.2, 2.8), gridspec_kw={"width_ratios": [4, 5, 4]}, sharey=True)
for ax, (title, col, cats) in zip(axs, PANELS):
    d = df.dropna(subset=[col])
    rate = d.groupby([col, "collision_year"])["KSI"].mean().unstack() * 100
    n = d.groupby(col).size()
    x = np.arange(len(cats)); w = 0.38
    ax.bar(x - w/2, [rate.loc[c, 2024] for c, _ in cats], w, label="2024", color="#4C78A8")
    ax.bar(x + w/2, [rate.loc[c, 2025] for c, _ in cats], w, label="2025", color="#E4572E")
    ax.axhline(overall, ls="--", lw=0.8, color="k", label=f"Overall ({overall:.1f}%)")
    ax.set_xticks(x)
    ax.set_xticklabels([f"{lab}\n{int(n[c]):,}" for c, lab in cats], fontsize=6.5)
    ax.set_title(title, fontsize=8.5); ax.set_ylim(0, 44)
    ax.spines[["top", "right"]].set_visible(False)
axs[0].set_ylabel("KSI rate (%)")
axs[0].legend(frameon=False, fontsize=6.5, loc="upper left", ncol=1)
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, "fig3_ksi_by_environment.png"), dpi=300)
plt.show()